<div class='heading'>
    <div style='float:left;'><h1>CPSC 8810 Machine Learning for Graphs</h1></div>
     <img style="float: right; padding-right: 10px" width="100" src="https://raw.githubusercontent.com/bsethwalker/clemson-cs4300/main/images/clemson_paw.png"> </div>
     </div>

**Clemson University**<br>
**Fall 2025**<br>
**Instructor(s):** Aaron Masino <br>

## Homework 3: Graph Neural Networks
This homework is intended to assess your knowledge of core elements of graph neural networks in the messge passing framework, application of graph neural networks, and elements of the PyTorch Geometric implemenation of GNN methods as introduced during the in-class lectures and labs. You may wish to refer to the course lectures and labs while completing this assignment.

**Unless otherwise noted in the problem instructions you may use any of the following Python libraries to complete the exercises:**
- numpy, scipy
- scikit-learn
- matplotlib, seaborn, pygraphviz
- PyTorch, PyTorch Geometric
- NetworkX

**Test code:** You may add test code after an exercise to evaluate your code. This is optional.


In [ ]:
# Google Colab setup
# mount the google drive - this is necessary to access supporting src
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/ml4g/data"

# Create output directory
import os
from pathlib import Path

# data directory
data_dir = Path('"/content/drive/MyDrive/Colab Notebooks/ml4g/data')
data_dir.mkdir(parents=True, exist_ok=True)

dir_lightning = Path(os.path.join(data_dir, "lightning"))
dir_lightning.mkdir(parents=True, exist_ok=True)

In [ ]:
# install missing libraries (this may take several minutes)
!pip3 install torch_geometric
!pip install lightning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
from IPython import display
from pathlib import Path
import os

from torch_geometric.datasets import TUDataset, FakeDataset
from torch_geometric.seed import seed_everything
from torch_geometric.utils import to_networkx
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F

# PyTorch Lightning
import lightning as L
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch import seed_everything
import lightning.pytorch.trainer as trainer

import torch
from torch import nn as tnn
from torch.utils.data import random_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc)

import torchmetrics as TM
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 123456
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# data directory
data_dir = Path('../data')
data_dir.mkdir(parents=True, exist_ok=True)

dir_lightning = Path(os.path.join(data_dir, "lightning"))
dir_lightning.mkdir(parents=True, exist_ok=True)

# Part 1 Permutation Invariance and Equivariance

## Exercise 1: Permutation Invariance (2 points)

In this exercise you are asked to implement a method that tests if a function $f$ is permutation invariant for the permutation $perm$ on the input $x$. Assume that:
-    $x\in \mathbb{R}^{NXD}$, i.e. $x$ is a $N \times D$ dimensional real matrix
-    $perm\in \mathbb{I}^{N}$ is a list of integer indices that represent the permutation of the rows of $x$. For example if `x=[[a,b,c], [d,e,f]]` and `perm=[1,0]` then `x[perm]=[[d,e,f], [a,b,c]]`
-    $f\in \mathbb{R}^{NXD}\to \mathbb{R}^{D}$ is mapping from an $N \times D$ dimensional real matrix to a  $D$ dimensional vector

In the code cell below, complete the implmentation of the `is_permutation_invariant`. The function should return `True` if $f$ is permutation invariant for the permutation $perm$ on the input $x$ and `False` otherwise.

In [ ]:
def is_permutation_invariant(f, x, perm, atol=1e-6):
    """
    Check if function f is permutation invariant on the input tensor x with respect to the permutation perm.

    Args:
        f: A callable that maps and N x D real matrix to a D dimensional vector.
        x: Input tensor of shape (N, D).
        perm: A permutation tensor of shape (N,) that defines the permutation.
        atol: Absolute tolerance for output equality check.
    Returns:
        bool: True if f is permutation invariant on x with respect to perm.
    """
    N, D = x.shape
    if perm.shape[0] != N:
        raise ValueError("Permutation length must match the number of rows in x.")
    if f(x).shape != (D,):
        raise ValueError("Function f must return a vector of shape (D,).")

    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


# tests for function is_permutation_invariant
x = torch.tensor([[1.0, 2.0], [4.0, 5.0], [7.0, 8.0]])
perm = torch.tensor([2, 0, 1])
assert is_permutation_invariant(lambda x: x.sum(dim=0), x, perm)
row_scale = torch.tensor([[1], [2], [3]])
assert not is_permutation_invariant(lambda x: (x * row_scale).sum(dim=0), x, perm)

## Exercise 2: Permutation Equivariance (2 points)

In this exercise you are asked to implement a method that tests if a function $f$ is permutation equivariant for the permutation $perm$ on the input $x$. Assume that:
-    $x\in \mathbb{R}^{NXD}$, i.e. $x$ is a $N \times D$ dimensional real matrix
-    $perm\in \mathbb{I}^{N}$ is a list of integer indices that represent the permutation of the rows of $x$. For example if `x=[[a,b,c], [d,e,f]]` and `perm=[1,0]` then `x[perm]=[[d,e,f], [a,b,c]]`
-    $f\in \mathbb{R}^{NXD}\to \mathbb{R}^{NXD}$ is mapping from an $N \times D$ dimensional real matrix to a  $N \times D$ dimensional vector

In the code cell below, complete the implmentation of the `is_permutation_invariant`. The function should return `True` if $f$ is permutation invariant for the permutation $perm$ on the input $x$ and `False` otherwise.

In [ ]:
def is_permutation_equivariant(f, x, perm, atol=1e-6):
    """
    Check if function f is permutation equivariant on the input tensor x with respect to the permutation perm.

    Args:
        f: A callable that maps an N x D real matrix to an N x D matrix.
        x: Input tensor of shape (N, D).
        perm: A permutation tensor of shape (N,) that defines the permutation.
        atol: Absolute tolerance for output equality check.
    Returns:
        bool: True if f is permutation equivariant on x with respect to perm.
    """
    N, D = x.shape
    if perm.shape[0] != N:
        raise ValueError("Permutation length must match the number of rows in x.")
    if f(x).shape != (N, D):
        raise ValueError("Function f must return a matrix of shape (N, D).")

    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

x = torch.tensor([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]])
perm = torch.tensor([2, 0, 1])
assert is_permutation_equivariant(torch.exp, x, perm)
row_scale = torch.tensor([[1], [2], [3]])
assert not is_permutation_equivariant(lambda x: torch.exp(x * row_scale), x, perm)

# Part 2: Understanding PyTorch Geometric Data Structures

## Exercise 3: Number of classes (2 points)

In this exercise you are asked to implement the `get_num_classes` method that returns the number of classes for an input PyTorch Geometric Data object. The method takes as input
-    `pyg_data` : an instance of the [torch_geometric.data.Data](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.data.Data.html#torch_geometric.data.Data) class.

The method should return the number of unique class labels in the data or None if there are no class labels. Assume that class labels, if present, are in the `y` element of the data where `y` is either a 1D tensor of labels for each sample (a node or a graph) **OR** a 2D tensor of one-hot labels (i.e., if there are 3 classes, each sample is labeled with one of `[1, 0, 0]`,  `[0,1,0]`, or `[0, 0, 1]`).

**NOTE** Your method should only use the `y` element in the Data. Do not use methods in the Datas class.

In [ ]:
def get_num_classes(pyg_data):
    num_classes = None
    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%
    return num_classes

## Exercise 4: Number of node features (2 points)

In this exercise you are asked to implement the `get_num_features` method that returns the number of node features for an input PyTorch Geometric Data object. The method takes as input
-    `pyg_data` : an instance of the [torch_geometric.data.Data](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.data.Data.html#torch_geometric.data.Data) class.

The method should return the number of features assigned to each node in the Data object or None if there are no node features. Assume that node features, if present, are in the `x` element of the data where `x` is 2D tensor where the first dimension refers to the sample and the second refers to the features.

**NOTE** Your method should only use the `x` element in the Data object. Do not use methods in the Data class.

In [ ]:
def get_num_features(pyg_data:
    num_features = None
    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%



    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%
    return num_features

## Exercise 5: Class label for graph $i$ (1 point)
In this exercise you are asked to implement the `get_class` method that returns the class label if it exists for an input PyTorch Geometric Dataset and index. The method takes as input
-    `pyg_dataset` : an instance of the [torch_geometric.data.Dataset](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.data.Dataset.html#torch_geometric.data.Dataset) class.
-    `idx` : an integer index for a node or graph

The method should return an intger indicating the class for the node or graph with index `idx` if it exists and raise a `ValueError` if `idx` is out of bounds. You may assume the following:
-    If there is only 1 graph in the dataset, the index refers to a node and the y values are node class labels, otherwise the index refers to a graph and the y values refer to graph class labels    
-    Class label integer are 0 based.
-    Class labels, if present, are in the `y` element of the dataset where `y` is either a 1D tensor of labels for each graph or node **OR** a 2D tensor of one-hot labels (i.e., if there are 3 classes, each sample is labeled with one of `[1, 0, 0]`,  `[0,1,0]`, or `[0, 0, 1]`).

**NOTE** Your method should only use the `y` element in the DataSet. Do not use methods in the Dataset class. <br/>
**HINT** For a one element tensor, `t`, the value can be accessed with `t.item()`.

In [ ]:
def get_class(pyg_dataset, idx):
    class_label = None
    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%
    return class_label

## Exercise 6: Number of nodes in graph $i$ (1 point)
In this exercise you are asked to implement the `get_node_count` method that returns the number of nodes for the input PyTorch Geometric Dataset and graph index. The method takes as input
-    `pyg_dataset` : an instance of the [torch_geometric.data.Dataset](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.data.Dataset.html#torch_geometric.data.Dataset) class.
-    `idx` : an integer index for graph

The method should return an intger indicating the number of nodes for graph with index `idx` if it exists and raise a `ValueError` if `idx` is out of bounds.

**NOTE** Your method should only use the `x` element for the specified graph in the `pyg_dataset` object. Do not use methods in the Dataset class. <br/>

In [ ]:
def get_node_count(pyg_dataset, idx):
    node_count = None
    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%
    return node_count

## Exercise 7: Number of edges in graph $i$ (1 point)
In this exercise you are asked to implement the `get_edge_count` method that returns the number of edges for the input PyTorch Geometric Dataset and graph index. The method takes as input
-    `pyg_dataset` : an instance of the [torch_geometric.data.Dataset](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.data.Dataset.html#torch_geometric.data.Dataset) class.
-    `idx` : an integer index for graph

The method should return an intger indicating the number of edges for graph with index `idx` if it exists and raise a `ValueError` if `idx` is out of bounds.

**NOTE** Your method should only use the `edge_index` element for the specified graph in the `pyg_dataset` object. Do not use methods in the Dataset class. <br/>

In [ ]:
def get_edge_count(pyg_dataset, idx):
    edge_count = None
    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%
    return edge_count

# Part 3: Building a graph neural network for node classification
In this section, you are asked to complete a series of exercises that will develop and evaluate a GNN model to predict the class label for input graphs from the `PROTEINS` dataset from `TUDataset`. Each protein is labeled as either an `enzyme` or `non-enzyme`, hence this is a binary classification task.  First, let's load the dataset and print the summary statistics. Note, each graph in the `proteins` dataset includes `x` (node features), `y` (graph label), `edge_index` (indices of connected nodes).

In [ ]:
# load the Protein dataset
#%%%%%%% DO NOT MODIFY THIS CODE %%%%%%%%%%%%%
proteins = TUDataset(root=f'{data_dir}', name='PROTEINS')
protein_class_map = {0: 'enzyme', 1: 'non-enzyme'}
proteins.print_summary()

## Exercise 8: Data splitting (3 points)
The `proteins` dataset does **not** include masks for data splitting. In the exercise below, you are asked to complete the `split_graph_data` method to create training, validation, and test splits of the graphs in the `proteins` dataset. The method takes as input:
-    `pyg_dataset` : an instance of the [torch_geometric.data.Dataset](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.data.Dataset.html#torch_geometric.data.Dataset) class.
-    `train_ratio` : the fraction of graphs that should be used for training
-    `val_ration` : the fraction of graphs that should be used for validation

Your method shold return the three data splits. **HINT** You should use the `torch.utils.data.random_split` method, which has already been imported. For reproducibility, you should create `torch.Generator()` object, call its `manual_seed` method, and pass it to the `random_split` method.

In [ ]:
def split_graph_data(pyg_dataset, train_ratio=0.7, val_ratio=0.1, random_state=RANDOM_STATE):
    train_dataset = None
    val_dataset = None
    test_dataset = None
    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


    # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

    return train_dataset, val_dataset, test_dataset

### Lightning data module
The code below creates a PyTorch Lightning Data Module for the `proteins` dataset which will be used to support model training and evaluation.

The cell uses the `split_graph_data` you created above. You should execute this cell and make sure that it executes properly.

### Please do **not** modify this code.

In [ ]:
# %%%%%%%%%%%%%%%%% DO NOT MODIFY THIS CODE %%%%%%%%%%%%%%%%%%%
class ProteinsDataModule(L.LightningDataModule):
    def __init__(self, dataset, batch_size=32, train_ratio = 0.7, val_ratio = 0.1, shuffle = True,
                 class_name_map = None):
        super().__init__()
        self.batch_size = batch_size
        self.val_ratio = val_ratio
        self.train_ratio = train_ratio
        self.class_name_map = class_name_map
        self.num_classes = None
        self.shuffle = shuffle
        self.dataset = dataset
        self.num_features = dataset.num_features
        self.num_classes = dataset.num_classes

        self.train_dataset, self.val_dataset, self.test_dataset = split_graph_data(
            self.dataset, train_ratio=self.train_ratio,
            val_ratio=self.val_ratio, random_state=RANDOM_STATE)

        self._train_dataloader = DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=self.shuffle)
        self._val_dataloader = DataLoader(self.val_dataset, batch_size=self.batch_size)
        self._test_dataloader = DataLoader(self.test_dataset, batch_size=self.batch_size)

    def train_dataloader(self):
        return self._train_dataloader

    def test_dataloader(self):
        return self._test_dataloader

    def val_dataloader(self):
        return self._val_dataloader

dm = ProteinsDataModule(proteins, batch_size=32, class_name_map=protein_class_map)
loader = dm.train_dataloader()

cnt = 0
max_cnt = 1
for step, data in enumerate(loader):
    print(f'Step {step + 1}:')
    print('=======')
    print(f'Number of graphs in the current batch: {data.num_graphs}')
    print(data)
    print()
    cnt += 1
    if cnt == max_cnt:
        break

print(dm.num_features, dm.num_classes)

## Excercise 9: Create the graph encoder module (3 points)
In this exercise, you are asked to implement the `GCNSkipEncoder` module by completing the `__init__` and `forward` methods. The model architecture should include `H = num_hidden_layers` graph convolutional layers. The layers should be added to the `tnn.ModuleList`. Except for the first layer, all other layers should have the same embedding size for the nodes. In the `forward` method, each graph convolution layer should be part of a graph convolution **block** that includes the graph convolution layer and a `ReLU` nonlinear activation. Additionally, your model should include skip connections from each layer to the final layer where the embeddings at each layer should be summed.

**HINT** Refer to lab 04, section 2.4

In [ ]:
class GCNSkipEncoder(tnn.Module):
    def __init__(self, node_feature_dim, num_hidden_layers, hidden_channels):
        super().__init__()
        self.hidden_layers = tnn.ModuleList()

        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

    def forward(self, x, edge_index):
        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

# Test code for the encoder
encoder = GCNSkipEncoder(dm.num_features, 2, 8)

# Test the encoder
sample_batch = next(iter(dm.train_dataloader()))
print("input shape",sample_batch)
encoder_output = encoder(sample_batch.x, sample_batch.edge_index)
print("Output shape:", encoder_output.shape)

## Excercise 10: Create the prediction head for binary classification (3 points)
In this exercise, you are asked to implement the `BinaryPredictionHead` module by completing the `__init__` and `forward` methods. The model should should use global mean pooling to aggregate the node embeddings for the nodes in a graph which will be generated by the encoder. Recall, this is binary classification, so your model should generate only a single output representing the logit for the postive class. Do **not** apply a sigmoid as this will be addressed in the loss function.

**HINT** Refer to lab 04, section 2.4

In [ ]:
class BinaryPredictionHead(tnn.Module):
    def __init__(self, hidden_channels):
        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

    def forward(self, x, batch):
        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%

        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


## Exercise 11: PyTorch Lighting model implementation (5 points)

In this exercise, you are asked to implement the PyTorch Lightning Module below by completing the `forward` and `training_step` methods. For the `training_step` method, use the `F.binary_cross_entropy_with_logits` method to calculate the loss. You will need to apply the `squeeze` method to the logits and cast the `y` values to float when passing to the loss function.

**HINT** See lab 04, section 2.5

In [ ]:
class GraphClassifier(L.LightningModule):
    def __init__(self, encoder, pred_head, num_classes):
        super().__init__()
        # model layers
        self.encoder = encoder
        self.pred_head = pred_head

        # validation metrics
        self.val_metrics_tracker = TM.wrappers.MetricTracker(TM.MetricCollection([TM.classification.BinaryAccuracy()]))
        self.validation_step_outputs = []
        self.validation_step_targets = []

        # test metrics
        self.test_roc = TM.classification.BinaryROC()
        self.test_auroc = TM.classification.BinaryAUROC()
        self.test_cm = TM.classification.BinaryConfusionMatrix()
        self.test_metrics_tracker = TM.wrappers.MetricTracker(TM.MetricCollection([TM.classification.BinaryAccuracy(),
                                                            self.test_roc, self.test_auroc, self.test_cm]), maximize=True)
        self.test_step_outputs = []
        self.test_step_targets = []

    def forward(self, databatch):
        x = None
        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%
        return x

    def training_step(self, databatch, batch_idx):
        loss = None
        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%


        # %%%%%%%%%%%%%%%%%%% YOUR CODE HERE %%%%%%%%%%%%%%%%%%%
        return loss

    def validation_step(self, databatch, batch_idx):
        y = databatch.y
        logits = self.forward(databatch)
        loss = F.binary_cross_entropy_with_logits(logits.view(-1), y.float())
        self.log('val_loss', loss, on_step=True, on_epoch=True)

        # store the outputs and targets for the epoch end step
        self.validation_step_outputs.append(logits)
        self.validation_step_targets.append(y)
        return loss

    def on_validation_epoch_end(self):
        # stack all the outputs and targets into a single tensor
        all_preds = torch.vstack(self.validation_step_outputs)
        all_targets = torch.hstack(self.validation_step_targets)

        # compute the metrics
        loss = F.binary_cross_entropy_with_logits(all_preds.view(-1), all_targets.float())
        self.val_metrics_tracker.increment()
        self.val_metrics_tracker.update(all_preds.squeeze(), all_targets.squeeze())
        self.log('val_loss_epoch_end', loss)

        # clear the validation step outputs
        self.validation_step_outputs.clear()
        self.validation_step_targets.clear()

    def test_step(self, databatch, batch_idx):
        y = databatch.y
        logits = self.forward(databatch)
        loss = F.binary_cross_entropy_with_logits(logits.view(-1), y.float())
        self.log('test_loss', loss, on_step=True, on_epoch=True)
        self.test_step_outputs.append(logits)
        self.test_step_targets.append(y)
        return loss

    def on_test_epoch_end(self):
        all_preds = torch.vstack(self.test_step_outputs)
        all_targets = torch.hstack(self.test_step_targets)

        loss = F.binary_cross_entropy_with_logits(all_preds.view(-1), all_targets.float())
        self.test_metrics_tracker.increment()
        self.test_metrics_tracker.update(all_preds.squeeze(), all_targets.squeeze())
        self.log('test_loss_epoch_end', loss)

        # clear the test step outputs
        self.test_step_outputs.clear()
        self.test_step_targets.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

### Model training
The code below can be used to train your model. You should use this code to test if your code in previous sections is working as you expect. Additionally, you may modify the `hidden_dim`, `layers`, `max_epochs`, and `patience` parameters to see how they affect model performance.

In [ ]:
seed_everything(RANDOM_STATE)
dm = ProteinsDataModule(proteins, batch_size=32, class_name_map=protein_class_map)
hidden_dim = 64
layers = 2
max_epochs = 400
patience = 25
encoder = GCNSkipEncoder(dm.num_features, layers, hidden_dim)
pred_head = BinaryPredictionHead(hidden_dim)
model = GraphClassifier(encoder, pred_head, num_classes=dm.num_classes)

trainer = L.Trainer(default_root_dir=dir_lightning,
                    max_epochs=max_epochs,
                    callbacks=[EarlyStopping(monitor="val_loss_epoch_end", mode="min", patience=patience)])
trainer.fit(model=model, train_dataloaders=dm.train_dataloader(), val_dataloaders=dm.val_dataloader())

## Model Performance Evaluation
The code below can be used to evaluate your model performance on the test set.

In [ ]:
trainer.test(model=model, dataloaders=dm.test_dataloader())
rslt = model.test_metrics_tracker.compute()


In [ ]:
device = torch.device("cpu")   #"cuda:0"
model.eval()
y_true=[]
y_pred=[]
with torch.no_grad():
    for databatch in dm.test_dataloader():
        y = databatch.y
        pred = torch.sigmoid(model(databatch)).squeeze(1)>0.5
        for i in range(len(pred)):
            y_true.append(y[i].item())
            y_pred.append(pred[i].item())

print(classification_report(y_true,y_pred,target_names=list(dm.class_name_map.values()),digits=4))

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

cmp = sns.heatmap(rslt['BinaryConfusionMatrix'], annot=True, fmt='d', cmap='Blues', ax=axes[0])
cmp.set_xlabel('Predicted Label')
cmp.set_xticklabels(dm.class_name_map.values(), rotation=45)
cmp.set_yticklabels(dm.class_name_map.values(), rotation=0)
cmp.set_ylabel('Actual Label');

fpr, tpr, thresholds = rslt['BinaryROC']
axes[1].plot(fpr, tpr, label='Model ROC')
plt.xlabel('False Positive Rate')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid()